# 02) ML Preprocessing & Dataset Preparation

This file starts from `student_timeline_part2.csv` and prepares leakage-safe datasets for the LSTM.

In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("..").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ML_DIR = PROCESSED_DIR / "ml"
ML_DIR.mkdir(parents=True, exist_ok=True)

TIMELINE_PATH = PROCESSED_DIR / "student_timeline_part2.csv"

print("Timeline:", TIMELINE_PATH)
print("ML output:", ML_DIR)

In [ ]:
student_timeline_part2 = pd.read_csv(TIMELINE_PATH)

print("Shape:", student_timeline_part2.shape)
print("Final result:")
print(student_timeline_part2["final_result"].value_counts(dropna=False))

# 1) Data Audit

In [ ]:
sequence_cols = [
    "id_student",
    "code_module",
    "code_presentation",
]

checkpoint_keys = [
    "id_student",
    "code_module",
    "code_presentation",
    "id_assessment",
]

print("Duplicate checkpoints:", student_timeline_part2.duplicated(
    subset=checkpoint_keys
).sum())

print("Bad time order:", (
    student_timeline_part2
    .sort_values(sequence_cols + ["time_step"])
    .groupby("sequence_id")["time_step"]
    .apply(lambda x: not x.is_monotonic_increasing)
    .sum()
))

print("Target values:")
print(student_timeline_part2["target"].value_counts())

# 2) Prediction Steps & Target

In [ ]:
student_timeline_part2["is_exam"] = (
    student_timeline_part2["assessment_type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("exam")
    .astype("int8")
)

# Early detection uses non-Exam checkpoints.
student_timeline_part2["is_prediction_step"] = (
    student_timeline_part2["is_exam"] == 0
).astype("int8")

student_timeline_part2["previous_score_observed"] = (
    student_timeline_part2
    .sort_values(sequence_cols + ["time_step"])
    .groupby(sequence_cols)["score_observed"]
    .shift(1)
)

student_timeline_part2["previous_score_available"] = (
    student_timeline_part2["previous_score_observed"]
    .notna()
    .astype("int8")
)

print(student_timeline_part2[[
    "sequence_id", "time_step", "score_observed",
    "previous_score_observed", "previous_score_available",
    "is_exam", "is_prediction_step"
]].head(20))

# 3) Sequence Split

Students are the groups. The same student cannot appear in different splits.

In [ ]:
sequence_meta = (
    student_timeline_part2
    .groupby(["sequence_id", "id_student"], as_index=False)
    .agg(
        target=("target", "first"),
        has_prediction_step=("is_prediction_step", "max"),
        sequence_length=("time_step", "max"),
    )
)

eligible_sequences = sequence_meta[
    sequence_meta["has_prediction_step"] == 1
].copy()

print("Total sequences:", len(sequence_meta))
print("Eligible sequences:", len(eligible_sequences))

In [ ]:
N_SPLITS = 7

sgkf = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

folds = list(
    sgkf.split(
        eligible_sequences[["sequence_id"]],
        eligible_sequences["target"],
        groups=eligible_sequences["id_student"],
    )
)

_, test_idx = folds[0]
_, validation_idx = folds[1]

test_sequences = set(
    eligible_sequences.iloc[test_idx]["sequence_id"].astype(int)
)

validation_sequences = set(
    eligible_sequences.iloc[validation_idx]["sequence_id"].astype(int)
)

all_sequences = set(eligible_sequences["sequence_id"].astype(int))
train_sequences = all_sequences - test_sequences - validation_sequences

print("Train sequences:", len(train_sequences))
print("Validation sequences:", len(validation_sequences))
print("Test sequences:", len(test_sequences))

In [ ]:
split_students = {}
for name, seqs in [
    ("train", train_sequences),
    ("validation", validation_sequences),
    ("test", test_sequences),
]:
    split_students[name] = set(
        eligible_sequences[
            eligible_sequences["sequence_id"].isin(seqs)
        ]["id_student"].astype(int)
    )

print("Student overlap:")
print("train & validation:", len(split_students["train"] & split_students["validation"]))
print("train & test:", len(split_students["train"] & split_students["test"]))
print("validation & test:", len(split_students["validation"] & split_students["test"]))

## 3.1 Split Summary

In [ ]:
split_summary = []

for name, seqs in [
    ("train", train_sequences),
    ("validation", validation_sequences),
    ("test", test_sequences),
]:
    meta = eligible_sequences[
        eligible_sequences["sequence_id"].isin(seqs)
    ]

    rows = student_timeline_part2[
        student_timeline_part2["sequence_id"].isin(seqs)
        & (student_timeline_part2["is_prediction_step"] == 1)
    ]

    split_summary.append({
        "split": name,
        "sequences": len(meta),
        "students": meta["id_student"].nunique(),
        "prediction_rows": len(rows),
        "positive_sequences": int(meta["target"].sum()),
        "positive_sequence_ratio": float(meta["target"].mean()),
        "positive_rows": int(rows["target"].sum()),
        "positive_row_ratio": float(rows["target"].mean()),
    })

split_summary_df = pd.DataFrame(split_summary)
print(split_summary_df)

# 4) Model Features

In [ ]:
CATEGORICAL_FEATURES = [
    "code_module",
    "code_presentation",
    "assessment_type",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
]

ACTIVITY_FEATURES = sorted([
    col for col in student_timeline_part2.columns
    if col.endswith("_cumulative_clicks")
    and col != "cumulative_clicks"
])

NUMERIC_FEATURES = [
    "weight",
    "cumulative_weight",
    "module_presentation_length",

    "previous_score_observed",
    "previous_score_available",
    "score_available",
    "submission_offset_days",
    "is_banked",

    "delta_days",
    "module_progress",

    "cumulative_clicks",
    "active_days",
    "days_since_last_activity",
    "has_activity",
    "recent_7d_clicks",
    "recent_14d_clicks",

    "num_of_prev_attempts",
    "studied_credits",
] + ACTIVITY_FEATURES

FEATURE_COLUMNS = CATEGORICAL_FEATURES + NUMERIC_FEATURES

missing_required = [
    col for col in FEATURE_COLUMNS
    if col not in student_timeline_part2.columns
]

if missing_required:
    raise ValueError(
        f"Missing required model columns: {missing_required}"
    )

print("Categorical features:", len(CATEGORICAL_FEATURES))
print("Numeric features:", len(NUMERIC_FEATURES))
print("Raw model features:", len(FEATURE_COLUMNS))

# 5) Preprocessing

The preprocessor is fitted on TRAIN only.

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, NUMERIC_FEATURES),
        ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

train_fit_df = student_timeline_part2[
    student_timeline_part2["sequence_id"].isin(train_sequences)
    & (student_timeline_part2["is_prediction_step"] == 1)
].copy()

preprocessor.fit(train_fit_df[FEATURE_COLUMNS])

feature_names = preprocessor.get_feature_names_out().tolist()

print("Transformed feature count:", len(feature_names))

joblib.dump(
    preprocessor,
    ML_DIR / "preprocessor.joblib",
)

with open(ML_DIR / "feature_names.json", "w", encoding="utf-8") as f:
    json.dump(feature_names, f, indent=2)

# 6) Transforming Splits

In [ ]:
META_COLUMNS = [
    "sequence_id",
    "time_step",
    "target",
    "is_prediction_step",
    "id_student",
    "id_assessment",
    "code_module",
    "code_presentation",
]

def transform_split(sequence_ids):
    split_df = student_timeline_part2[
        student_timeline_part2["sequence_id"].isin(sequence_ids)
    ].copy()

    split_df = split_df.sort_values(
        ["sequence_id", "time_step"]
    )

    transformed = preprocessor.transform(
        split_df[FEATURE_COLUMNS]
    )

    transformed_df = pd.DataFrame(
        transformed,
        columns=feature_names,
        index=split_df.index,
    )

    return pd.concat(
        [
            split_df[META_COLUMNS].reset_index(drop=True),
            transformed_df.reset_index(drop=True),
        ],
        axis=1,
    )

train_processed = transform_split(train_sequences)
validation_processed = transform_split(validation_sequences)
test_processed = transform_split(test_sequences)

print("Train:", train_processed.shape)
print("Validation:", validation_processed.shape)
print("Test:", test_processed.shape)

# 7) Processed Data Validation

In [ ]:
def check_processed(df, name):
    feature_block = df[feature_names].to_numpy(dtype=np.float64)

    print(f"\n{name}")
    print("NaN:", int(np.isnan(feature_block).sum()))
    print("Inf:", int(np.isinf(feature_block).sum()))
    print(
        "Duplicate sequence/time:",
        df.duplicated(
            subset=["sequence_id", "time_step"]
        ).sum()
    )

check_processed(train_processed, "TRAIN")
check_processed(validation_processed, "VALIDATION")
check_processed(test_processed, "TEST")

# 8) Baseline Model

Logistic Regression is used as a non-sequential reference model.

In [ ]:
def evaluate_binary(y_true, probabilities, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    probabilities = np.asarray(probabilities)
    y_pred = (probabilities >= threshold).astype(int)

    return {
        "roc_auc": (
            roc_auc_score(y_true, probabilities)
            if len(np.unique(y_true)) == 2
            else np.nan
        ),
        "pr_auc": (
            average_precision_score(y_true, probabilities)
            if y_true.sum() > 0
            else np.nan
        ),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "brier": brier_score_loss(y_true, probabilities),
    }

train_rows = train_processed[train_processed["is_prediction_step"] == 1]
validation_rows = validation_processed[validation_processed["is_prediction_step"] == 1]
test_rows = test_processed[test_processed["is_prediction_step"] == 1]

baseline = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs",
)

baseline.fit(
    train_rows[feature_names],
    train_rows["target"],
)

validation_prob = baseline.predict_proba(
    validation_rows[feature_names]
)[:, 1]

test_prob = baseline.predict_proba(
    test_rows[feature_names]
)[:, 1]

precision, recall, thresholds = precision_recall_curve(
    validation_rows["target"],
    validation_prob,
)

f1_curve = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)

best_idx = int(np.nanargmax(f1_curve))
baseline_threshold = float(thresholds[best_idx])

validation_metrics = evaluate_binary(
    validation_rows["target"].to_numpy(),
    validation_prob,
    baseline_threshold,
)

test_metrics = evaluate_binary(
    test_rows["target"].to_numpy(),
    test_prob,
    baseline_threshold,
)

print("Baseline validation:", validation_metrics)
print("Baseline test:", test_metrics)

## 8.1 Saving Baseline Predictions

In [ ]:
baseline_test_predictions = test_rows[[
    "sequence_id",
    "time_step",
    "target",
    "id_student",
    "code_module",
    "code_presentation",
]].copy()

baseline_test_predictions["risk_probability"] = test_prob
baseline_test_predictions["prediction"] = (
    test_prob >= baseline_threshold
).astype(int)

baseline_test_predictions.to_csv(
    ML_DIR / "baseline_test_predictions_by_timestep.csv",
    index=False,
)

with open(ML_DIR / "baseline_metrics.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "threshold": baseline_threshold,
            "validation": validation_metrics,
            "test": test_metrics,
        },
        f,
        indent=2,
    )

baseline_test_predictions.head(20)

# 9) Exporting ML Files

In [ ]:
train_processed.to_csv(
    ML_DIR / "train_processed.csv",
    index=False,
)

validation_processed.to_csv(
    ML_DIR / "validation_processed.csv",
    index=False,
)

test_processed.to_csv(
    ML_DIR / "test_processed.csv",
    index=False,
)

sequence_manifest = eligible_sequences.copy()

sequence_manifest["split"] = np.select(
    [
        sequence_manifest["sequence_id"].isin(train_sequences),
        sequence_manifest["sequence_id"].isin(validation_sequences),
        sequence_manifest["sequence_id"].isin(test_sequences),
    ],
    ["train", "validation", "test"],
    default="excluded",
)

sequence_manifest.to_csv(
    ML_DIR / "sequence_manifest.csv",
    index=False,
)

metadata = {
    "seed": SEED,
    "at_risk_results": sorted(AT_RISK_RESULTS),
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "feature_count_after_encoding": len(feature_names),
    "split_strategy": "StratifiedGroupKFold_7_by_student",
    "early_detection_prediction_steps": "non-Exam checkpoints only",
    "current_score_used": False,
    "previous_score_used": True,
}

with open(ML_DIR / "preprocessing_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved ML files to:", ML_DIR)

# 10) Final ML Audit

In [ ]:
print("=== FINAL ML AUDIT ===")
print("Train sequences:", len(train_sequences))
print("Validation sequences:", len(validation_sequences))
print("Test sequences:", len(test_sequences))
print("Transformed features:", len(feature_names))

print("Student overlap:")
print(len(split_students["train"] & split_students["validation"]))
print(len(split_students["train"] & split_students["test"]))
print(len(split_students["validation"] & split_students["test"]))

for name, df in [
    ("train", train_processed),
    ("validation", validation_processed),
    ("test", test_processed),
]:
    block = df[feature_names].to_numpy(dtype=np.float64)
    print(
        name,
        "rows=", len(df),
        "nan=", int(np.isnan(block).sum()),
        "inf=", int(np.isinf(block).sum()),
    )

# ✅ What happened in this file?

We started from `student_timeline_part2.csv` and produced: split data, train-only preprocessing, an interpretable Logistic Regression baseline, and the ML-ready tables used by the LSTM.